# ML Testing & MLOps — 2026-11-07 — Building a Model Release Gate for a Wine-Quality Classifier

## Problem statement

You are the QA engineer responsible for deciding whether a new build of a "good wine" classifier is
safe to ship. The model predicts whether a red wine sample is high quality (expert rating ≥ 7) from
11 physicochemical lab measurements — acidity, sugar, sulfur dioxide, alcohol, etc. — using the real
**UCI Wine Quality (red)** dataset (1,599 samples, 11 numeric features, no missing values).

Your job in this notebook is **not** to find the most accurate model. It's to build the automated
**test suite** that gates every candidate model before it reaches production — the thing an SQA
engineer moving into ML would actually own. Any model, present or future, should have to pass through
these checks.

**What you should produce**, concretely:

1. **Data validation tests** — `test_schema`, `test_value_ranges`, `test_class_balance` — that catch a
   corrupted or drifted input batch (wrong dtype, out-of-range reading, collapsed minority class)
   *before* it reaches training or inference.
2. **An evaluation gate that resists the "accuracy trap"** — only ~13.6% of samples are "good" wine,
   so a model that always predicts "not good" scores ~86.6% accuracy with **zero** recall on the class
   that actually matters. Your gate must assert a minimum *recall on the minority class*, not just an
   accuracy threshold, and you must show numerically that the accuracy-only gate would have let the
   degenerate model through.
3. **A drift detector** using the Population Stability Index (PSI) on the `alcohol` feature, run once
   against a held-out batch from the *same* distribution (should pass) and once against a batch you
   deliberately shift (should fail) — proving the detector actually discriminates.
4. **A reproducibility test** — retraining with the same random seed must give byte-for-byte identical
   metrics; retraining with a different seed must stay within a small tolerance band of the original.

Fill in every `# TODO` in the starter cells below, run the checks, and compare your numbers against
the collapsed solution at the bottom.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## Load the data

Real dataset: UCI Wine Quality (red wine), fetched as a CSV mirror (the UCI archive itself isn't reachable
from this environment, so we pull from a well-known GitHub mirror of the same file — same 1,599 rows,
12 columns, no header row, so we supply the official UCI column names ourselves).


In [ ]:
WINE_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/winequality-red.csv"

COLUMNS = [
    "fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar", "chlorides",
    "free_sulfur_dioxide", "total_sulfur_dioxide", "density", "pH", "sulphates",
    "alcohol", "quality",
]

df = pd.read_csv(WINE_URL, header=None, names=COLUMNS)
df["quality_good"] = (df["quality"] >= 7).astype(int)
df.head()


## Part 1 — Data validation tests

Fill in `EXPECTED_RANGES` for the features left blank (look at `df.describe()` and pad the real min/max
a little so legitimate new readings aren't rejected), then implement the three test functions.


In [ ]:
# TODO: fill in the missing ranges (pad real min/max slightly). A few are done for you.
EXPECTED_RANGES = {
    "fixed_acidity": (4.0, 17.0),
    "volatile_acidity": (0.0, 2.0),
    "citric_acid": None,          # TODO
    "residual_sugar": None,       # TODO
    "chlorides": None,            # TODO
    "free_sulfur_dioxide": None,  # TODO
    "total_sulfur_dioxide": (0.0, 300.0),
    "density": (0.98, 1.01),
    "pH": None,                   # TODO
    "sulphates": None,            # TODO
    "alcohol": (8.0, 15.5),
}


def test_schema(frame: pd.DataFrame) -> list[str]:
    """Return a list of human-readable errors (empty list = pass).
    Check: all COLUMNS present, every feature column is a float dtype, no nulls anywhere."""
    # TODO
    raise NotImplementedError


def test_value_ranges(frame: pd.DataFrame) -> list[str]:
    """Return a list of 'column: N rows out of [lo, hi]' errors for every column in EXPECTED_RANGES."""
    # TODO
    raise NotImplementedError


def test_class_balance(frame: pd.DataFrame, target_col: str = "quality_good", min_minority_frac: float = 0.05):
    """Return (passed: bool, minority_fraction: float). Fail if the rarer class drops below min_minority_frac."""
    # TODO
    raise NotImplementedError


## Part 2 — Evaluation gate that resists the accuracy trap

Split the data, train a baseline `RandomForestClassifier`, and implement an evaluation function that
returns more than accuracy. Then explicitly build the "always predict majority class" baseline and
show what an accuracy-only gate would have done with it.


In [ ]:
FEATURE_COLS = [c for c in COLUMNS if c != "quality"]

X = df[FEATURE_COLS]
y = df["quality_good"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)


def train_model(X_tr, y_tr, seed: int = RANDOM_SEED) -> RandomForestClassifier:
    """Train and return a RandomForestClassifier. Use class_weight='balanced'."""
    # TODO
    raise NotImplementedError


def evaluate_model(model, X_te, y_te) -> dict:
    """Return a dict with keys: accuracy, precision, recall, f1, confusion_matrix (as a nested list)."""
    # TODO
    raise NotImplementedError


def test_minimum_recall(metrics: dict, threshold: float = 0.5) -> bool:
    """Gate: pass only if metrics['recall'] >= threshold."""
    # TODO
    raise NotImplementedError

# TODO: train `model`, call evaluate_model to get `metrics`, and print it.

# TODO: build `majority_preds` = an array of all zeros, same length as y_test, and compute its
# accuracy and recall with accuracy_score / recall_score to see what an accuracy-only gate would miss.


## Part 3 — Drift detection with PSI

Implement the Population Stability Index between a baseline distribution (`X_train['alcohol']`) and a
candidate batch. Run it once on an undrifted batch (`X_test['alcohol']`, should pass) and once on a
batch you shift by adding `+1.5` (plus a little noise) to every value (should fail).


In [ ]:
def compute_psi(expected: np.ndarray, actual: np.ndarray, buckets: int = 10) -> float:
    """Population Stability Index between two 1-D arrays, using `buckets` quantile bins of `expected`."""
    # TODO
    raise NotImplementedError


def test_no_significant_drift(psi_value: float, threshold: float = 0.2) -> bool:
    """Gate: pass only if psi_value < threshold."""
    # TODO
    raise NotImplementedError

# TODO: compute psi_no_drift using X_train['alcohol'] vs X_test['alcohol'].

# TODO: build a drifted batch (X_test['alcohol'] + 1.5 + small gaussian noise, seeded) and compute
# psi_drift against the same baseline. Print both PSI values and both gate results.


## Part 4 — Reproducibility test

A model build pipeline that isn't reproducible can't be trusted in CI: retraining on the exact same
data and seed must give identical metrics, and a different seed should only move the metrics a little.


In [ ]:
def test_reproducibility(X_tr, y_tr, X_te, y_te, seed_a: int, seed_b: int, tolerance: float = 0.1):
    """Train with seed_a and seed_b, evaluate both, and return:
    (same_seed_identical: bool, cross_seed_within_tolerance: bool, metrics_a: dict, metrics_b: dict)
    where same_seed_identical compares two models BOTH trained with seed_a, and
    cross_seed_within_tolerance checks |f1_a - f1_b| < tolerance for seed_a vs seed_b."""
    # TODO
    raise NotImplementedError

# TODO: call test_reproducibility with seed_a=RANDOM_SEED, seed_b=7 and print the results.


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

### Part 1 — data validation

```python
EXPECTED_RANGES = {
    "fixed_acidity": (4.0, 17.0),
    "volatile_acidity": (0.0, 2.0),
    "citric_acid": (0.0, 1.1),
    "residual_sugar": (0.5, 16.0),
    "chlorides": (0.0, 0.7),
    "free_sulfur_dioxide": (0.0, 80.0),
    "total_sulfur_dioxide": (0.0, 300.0),
    "density": (0.98, 1.01),
    "pH": (2.5, 4.2),
    "sulphates": (0.2, 2.1),
    "alcohol": (8.0, 15.5),
}

def test_schema(frame):
    errors = []
    for c in COLUMNS:
        if c not in frame.columns:
            errors.append(f"missing column {c}")
    for c in FEATURE_COLS:
        if not pd.api.types.is_float_dtype(frame[c]):
            errors.append(f"{c} not float")
    if frame.isnull().any().any():
        errors.append("nulls present")
    return errors

def test_value_ranges(frame):
    errors = []
    for col, (lo, hi) in EXPECTED_RANGES.items():
        bad = frame[(frame[col] < lo) | (frame[col] > hi)]
        if len(bad) > 0:
            errors.append(f"{col}: {len(bad)} rows out of [{lo},{hi}]")
    return errors

def test_class_balance(frame, target_col="quality_good", min_minority_frac=0.05):
    frac = frame[target_col].mean()
    minority_frac = min(frac, 1 - frac)
    return minority_frac >= min_minority_frac, minority_frac

print(test_schema(df))          # []
print(test_value_ranges(df))    # []
print(test_class_balance(df))   # (True, 0.136...)
```

On this dataset all three pass cleanly — 13.6% of samples are "good" wine, comfortably above a 5%
floor. The point isn't that these particular checks fail here; it's that they *would* catch a bad
batch (e.g. a sensor sending `pH` readings of 12, or a batch where "good" wine silently drops to 0.5%
of samples because of an upstream filter bug) before that batch ever reaches a model.

### Part 2 — evaluation gate vs. the accuracy trap

```python
def train_model(X_tr, y_tr, seed=RANDOM_SEED):
    model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=seed, class_weight="balanced")
    model.fit(X_tr, y_tr)
    return model

def evaluate_model(model, X_te, y_te):
    preds = model.predict(X_te)
    return {
        "accuracy": accuracy_score(y_te, preds),
        "precision": precision_score(y_te, preds),
        "recall": recall_score(y_te, preds),
        "f1": f1_score(y_te, preds),
        "confusion_matrix": confusion_matrix(y_te, preds).tolist(),
    }

def test_minimum_recall(metrics, threshold=0.5):
    return metrics["recall"] >= threshold

model = train_model(X_train, y_train)
metrics = evaluate_model(model, X_test, y_test)
print(metrics)
# {'accuracy': 0.9125, 'precision': 0.647, 'recall': 0.767, 'f1': 0.702, 'confusion_matrix': [[259, 18], [10, 33]]}

majority_preds = np.zeros_like(y_test)
print(accuracy_score(y_test, majority_preds), recall_score(y_test, majority_preds, zero_division=0))
# 0.8656 accuracy, 0.0 recall

print(test_minimum_recall(metrics))                                    # True  — real model passes
print(test_minimum_recall({"recall": 0.0}))                            # False — degenerate model is blocked
```

This is the whole point of the exercise: the degenerate "always say no" model scores **86.6% accuracy**
— higher than plenty of teams' release bar — while missing every single good wine (**0% recall**). An
accuracy-only gate ships it. A recall-gated test suite correctly rejects it while still passing the
real model (76.7% recall, 91.25% accuracy).

### Part 3 — PSI drift detection

```python
def compute_psi(expected, actual, buckets=10):
    breakpoints = np.percentile(expected, np.linspace(0, 100, buckets + 1))
    breakpoints[0] -= 1e-6
    breakpoints[-1] += 1e-6
    expected_pct = np.histogram(expected, breakpoints)[0] / len(expected)
    actual_pct = np.histogram(actual, breakpoints)[0] / len(actual)
    expected_pct = np.clip(expected_pct, 1e-6, None)
    actual_pct = np.clip(actual_pct, 1e-6, None)
    return np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))

def test_no_significant_drift(psi_value, threshold=0.2):
    return psi_value < threshold

baseline_alcohol = X_train["alcohol"].values
psi_no_drift = compute_psi(baseline_alcohol, X_test["alcohol"].values)
print(psi_no_drift, test_no_significant_drift(psi_no_drift))           # ~0.03, True

rng = np.random.default_rng(0)
drifted_alcohol = X_test["alcohol"].values + rng.normal(1.5, 0.3, size=len(X_test))
psi_drift = compute_psi(baseline_alcohol, drifted_alcohol)
print(psi_drift, test_no_significant_drift(psi_drift))                 # ~5.07, False
```

PSI stays tiny (~0.03, well under the conventional 0.1–0.2 "watch" band) when the test batch comes from
the same distribution, and explodes to ~5.07 — more than 25x the alert threshold — once alcohol content
is shifted by just +1.5 degrees. A detector that can't distinguish these two cases is useless; this one
clearly can, which is what you'd want to see in a PR that adds a new drift check.

### Part 4 — reproducibility

```python
def test_reproducibility(X_tr, y_tr, X_te, y_te, seed_a, seed_b, tolerance=0.1):
    model_a1 = train_model(X_tr, y_tr, seed=seed_a)
    model_a2 = train_model(X_tr, y_tr, seed=seed_a)
    metrics_a1 = evaluate_model(model_a1, X_te, y_te)
    metrics_a2 = evaluate_model(model_a2, X_te, y_te)
    same_seed_identical = metrics_a1 == metrics_a2

    model_b = train_model(X_tr, y_tr, seed=seed_b)
    metrics_b = evaluate_model(model_b, X_te, y_te)
    cross_seed_within_tolerance = abs(metrics_a1["f1"] - metrics_b["f1"]) < tolerance

    return same_seed_identical, cross_seed_within_tolerance, metrics_a1, metrics_b

same_ok, cross_ok, m_a, m_b = test_reproducibility(X_train, y_train, X_test, y_test, RANDOM_SEED, 7)
print(same_ok, cross_ok)   # True True
```

Same seed gives byte-for-byte identical metrics dicts (`RandomForestClassifier` with a fixed
`random_state` is fully deterministic given fixed data), and seed 7 lands within 0.1 F1 of seed 42
(0.702 vs 0.723). If a "reproducibility" run ever came back `same_seed_identical=False`, that's a bug
in the pipeline (unpinned randomness, non-deterministic data loading order, etc.) worth blocking a
release over — exactly the kind of regression this test exists to catch.

</details>
